In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt

## ??????? ??????? ????????

In [ ]:
def load_accelerometer_data(base_dir):
    """
    ?????????? ???? ? ????????????? ? ??????????.
    
    Args:
        base_dir (str): ???? ?? ????????? ??????????.
    
    Returns:
        DataFrame: ???? ? ??????? pandas DataFrame.
    """
    data = []
    for class_dir in os.listdir(base_dir):  # ????? ????????????
        class_path = os.path.join(base_dir, class_dir)
        if os.path.isdir(class_path):  # ?????????, ?? ?? ??????????
            for file in os.listdir(class_path):  # ????? ?????? ? ???????????
                file_path = os.path.join(class_path, file)
                if file.endswith('.csv'):
                    # ?????????? CSV
                    df = pd.read_csv(file_path)
                    df['class'] = class_dir  # ????????? ????? ?????
                    df['file_name'] = file  # ??? ????????????? ?????
                    data.append(df)
    
    # ??'??????? ???? ????? ? ???? DataFrame
    return pd.concat(data, ignore_index=True)

## ????? ????? ?? ???????? ?? ???? ????????????

In [ ]:
base_dir = "Hw5/data"
df = load_accelerometer_data(base_dir)

In [ ]:
df.shape

In [ ]:
df.groupby(by="class")['file_name'].nunique()

## ?????????? time domain features

In [ ]:
def calculate_time_features(df):
    features = {}
    
    # ?????? ???? ?????????????
    axes = ['accelerometer_X', 'accelerometer_Y', 'accelerometer_Z']
    
    for axis in axes:
        # ??????? ??????????? ???????
        features[f'{axis}_mean'] = np.mean(df[axis])
        features[f'{axis}_std'] = np.std(df[axis])
        features[f'{axis}_min'] = np.min(df[axis])
        features[f'{axis}_max'] = np.max(df[axis])
        features[f'{axis}_range'] = np.max(df[axis]) - np.min(df[axis])
        features[f'{axis}_mad'] = np.mean(np.abs(df[axis] - np.mean(df[axis])))
        
        # ????????? ??????????????
        features[f'{axis}_energy'] = np.sum(df[axis]**2)
        features[f'{axis}_skew'] = df[axis].skew()
        features[f'{axis}_kurtosis'] = df[axis].kurtosis()
    
    # ???????? ???????
    magnitude = np.sqrt(df['accelerometer_X']**2 + df['accelerometer_Y']**2 + df['accelerometer_Z']**2)
    features['magnitude_mean'] = np.mean(magnitude)
    features['magnitude_std'] = np.std(magnitude)
    features['magnitude_max'] = np.max(magnitude)
    features['magnitude_min'] = np.min(magnitude)
    
    return features

## ????????? ??????? ? ?????? ????????

In [ ]:
grouped = df.groupby(['file_name', 'class'])

time_features = []

for (file_name, class_label), group in grouped:
    features = calculate_time_features(group)
    features['file_name'] = file_name
    features['class'] = class_label
    time_features.append(features)

# ???????????? ?????? ????????? ? DataFrame
features_df = pd.DataFrame(time_features)

In [ ]:
features_df.head()

In [ ]:
features_df.shape

## ?????????? ?????????? ??????? ?? ?????????? ??????? ?? ??????????? ?? ??????? ???? ?? ??????????? ????????? ??????

In [ ]:
X = features_df.drop(columns=['file_name', 'class'])  # ?????????? ????????
y = features_df['class']

In [ ]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

## ????????: ?????????? ??? ? ????? ????????

In [ ]:
rf_model = RandomForestClassifier(random_state=42).fit(X_train, y_train)

In [ ]:
y_pred = rf_model.predict(X_test)

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

## ????????? ?????? ?????????? ?????

In [ ]:
# ????????? ?????????? ?????
feature_importances = rf_model.feature_importances_

# ????????? DataFrame ??? ????????????
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

# ????????????
plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='skyblue')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.title('Feature Importance')
plt.gca().invert_yaxis()  # ??????????? ???? Y ??? ????? ???????? ???????
plt.show()

## ???????? ???????????? ?????? ? ???????

In [ ]:
# ????????? ????? ?? ?????? ?????
low_importance_features = [
    'accelerometer_Y_kurtosis',
    'accelerometer_X_kurtosis',
    'accelerometer_Z_skew',
    'accelerometer_Y_skew',
    'accelerometer_Z_kurtosis',
    'accelerometer_Z_energy'
]

X_train_reduced = X_train.drop(columns=low_importance_features)
X_test_reduced = X_test.drop(columns=low_importance_features)

In [ ]:
X_train_reduced.shape[1]

In [ ]:
rf_reduced_model = RandomForestClassifier(random_state=42).fit(X_train_reduced, y_train)

In [ ]:
y_pred_reduced = rf_reduced_model.predict(X_test_reduced)

In [ ]:
accuracy_score(y_test, y_pred_reduced)

In [ ]:
print(classification_report(y_test, y_pred_reduced, target_names=label_encoder.classes_))

## ???????? ?????????? 6 ?????

In [ ]:
high_importance_features = [
    'accelerometer_Y_std',
    'accelerometer_Y_max',
    'accelerometer_Y_mad',
    'accelerometer_X_mad',
    'accelerometer_Y_mean',
    'accelerometer_X_energy'
]

X_train_6 = X_train[high_importance_features]
X_test_6 = X_test[high_importance_features]
X_train_6.head()

In [ ]:
rf_6_model = RandomForestClassifier(random_state=42).fit(X_train_6, y_train)

In [ ]:
y_pred_6 = rf_6_model.predict(X_test_6)

In [ ]:
accuracy_score(y_test, y_pred_6)

In [ ]:
print(classification_report(y_test, y_pred_6, target_names=label_encoder.classes_))

## __????????:__
??????? ???? ????????? ????? ?????? "?????????? ???" ???????? ?? ???????? ???????? ???????? ???????????? ????? stairs.

?????, ???? ???????? 6 ??????? ???????? ?????, ?? ?? ???????? ?????? ?? ????? ?? ??????

## ????????: SVM

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled[0]

## ???? "linear"

In [ ]:
svm_linear_model = SVC(kernel='linear', random_state=42).fit(X_train_scaled, y_train)

In [ ]:
y_pred = svm_linear_model.predict(X_test_scaled)
print(f"???????? ?????? SVM: {accuracy_score(y_test, y_pred)}")
print("???? ??? ????????????:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

## ???? "RBF"

In [ ]:
svm_rbf_model = SVC(kernel='rbf', random_state=42).fit(X_train_scaled, y_train)

In [ ]:
y_pred = svm_rbf_model.predict(X_test_scaled)
print(f"???????? ?????? SVM: {accuracy_score(y_test, y_pred)}")
print("???? ??? ????????????:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

## ??????????? ???????????????

In [ ]:
c_range = np.logspace(-0, 4, 8)
gamma_range = np.logspace(-4, 0, 8)
tuned_parameters = [
    {
        'kernel': ['rbf'],
        'C': c_range,
        'gamma': gamma_range
    },
    {
        'kernel': ['linear'],
        'C': c_range,
        'gamma': gamma_range
    }
]
svc = SVC()
clf = GridSearchCV(
    svc,
    param_grid=tuned_parameters,
    verbose=20,
    cv=4,
    scoring='explained_variance'
)
clf.fit(X_train_scaled, y_train)
print('Best score for data:', clf.best_score_)
print('Best C:', clf.best_estimator_.C)
print('Best Kernel:', clf.best_estimator_.kernel)
print('Best Gamma:', clf.best_estimator_.gamma)

In [ ]:
svm_best_model = clf.best_estimator_

In [ ]:
y_pred = svm_best_model.predict(X_test_scaled)
print(f"???????? ?????? SVM: {accuracy_score(y_test, y_pred)}")
print("???? ??? ????????????:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

## __????????:__
??????? ?????? svm ?? ???????????? ??????????? ????? ???????????? ? ??????? idle, running, walking, ????? ?? ???? ????? ? ?????????????? ????? stairs (82%). ?????? ?? ??? ?? ???'????? ? ????? ????????? ????????? ? ????????.

rbf ?????? svm ?? ???????????? ??????????? ??? ?????? ?????? ?????????, ??? ???????? ??????????? ????? ?, ???? ?? ???? ????? ????????? ???? stairs (83%).

????? ????????? ??????????? ??????????????? ? ???? ??????????, ? ????? ?????? ???????? ???? ????? ????????? ?? rbf, ?????? ???????? ??????????????: ????????? ????????????? ?? gamma, ????????? ???????? ?????? ? ?????? ????????????????: 
- C: 193.06977288832496
- Kernel: rbf
- Gamma: 0.0193069772888324

? ????????? ???????? ????????????? ?????? 0.9993221532318642 ? ???????? ????????????? ????? stairs: 98%, ??? ?? ??? ???? ????? ?????? ????????? ????????? ?? 100% ????????.

????, ??????? ??????? ??????????? ???????????? ??????????????, ?? ??????? ?????? ? ?????????? ????????????????, ??? ?????????? ???? ????????? ? ???????? 99,9%

## ?????????? ?????? svm ? ?????????? ?????.

?????? ?????????? ??? ????????? ???? ??????? ? ?????? ??????????????? ?? ??????? ? ????????, ???? ???? ?? ???????? 6 ??????? ???????? ?????, ??? ????????? ???????????? ?????? ?????? ? ?? ???????? ??????????? ?? ????? ?? ????????. ???????? ?????? ??????????? ???? (?? ????????? 6 ??????? ???????? ?????) ???? ???????? ???????? ????????????? ?????? 0.9992266047950503 ? ???????? ????????????? ???? ?????? ??????: ???? stairs = 98%, ???? ????? = 100%.

?????? svm ????????? ???????? ? ??????? ??????????? ??????????????? ?? ????????? ? ????????, ??? ??????? ????????? ?????? ????, ??? ?????? ???????? ??????????????. ???????? ?????? svm ???? ???????? ???????? ????????????? ?????? 0.9993221532318642 ? ???????? ????????????? ???? ?????? ??????: ???? stairs = 98%, ???? ????? = 100%.

?????, ????? ???????, ?? ?????? ?????????? ??? ????????? ?????? ????????? ??? ???????? ????? ?? ???????? ??????? ??????????????? ?? ????????? ????????, ????? ??? ????????? ????????????, ????? ????????? ?????? svm, ???? ?? ??????? ??????????? ??????? ??'????? ?????????, ??? ? ??????????? ????, ? ????????? ???????? ??????????? ???????? ????????? ????????? ??? ???? ???????.